# MiniOneRec：将 item ID 转换为 Semantic ID 数据集

本 notebook 调用仓库根目录的 `convert_dataset.py`。它读取预处理交互、商品标题与 `item_id → SID` 映射，写出供 instruction tuning 使用的 train / valid / test CSV 及 SID 信息表。

In [1]:
# 配置预处理数据目录、转换脚本和新的输出目录。
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path(r'D:/转码ing/MiniOneRec-main')
DATASET = 'Industrial_and_Scientific'
DATA_DIR = PROJECT_ROOT / 'data' / 'Amazon18_2016_10_2018_11' / DATASET
CONVERT_SCRIPT = PROJECT_ROOT / 'convert_dataset.py'

# 使用单独输出目录，避免与原始预处理文件或仓库示例数据混在一起。
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'Amazon18'
CATEGORY = DATASET

# None 表示保留全部验证集和测试集样本。
MAX_VALID_SAMPLES = None
MAX_TEST_SAMPLES = None
SEED = 42

# False 时若输出目录已经存在则停止，避免混入不同 SID 版本的结果。
ALLOW_OVERWRITE = False


In [2]:
# 在转换前检查输入文件覆盖关系、SID 映射完整性和输出目录状态。
import json
from collections import Counter

ITEM_FILE = DATA_DIR / f'{DATASET}.item.json'
INDEX_FILE = DATA_DIR / f'{DATASET}.index.json'
SPLIT_FILES = {
    split_name: DATA_DIR / f'{DATASET}.{split_name}.inter'
    for split_name in ['train', 'valid', 'test']
}

required_files = [CONVERT_SCRIPT, ITEM_FILE, INDEX_FILE, *SPLIT_FILES.values()]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('缺少以下输入文件：\n' + '\n'.join(map(str, missing_files)))

if OUTPUT_DIR.exists() and not ALLOW_OVERWRITE:
    raise FileExistsError(
        f'输出目录已存在：{OUTPUT_DIR}\n'
        '请先检查已有结果；确认需要覆盖后，将 ALLOW_OVERWRITE 改为 True。'
    )

with ITEM_FILE.open('r', encoding='utf-8') as file:
    items = json.load(file)

with INDEX_FILE.open('r', encoding='utf-8') as file:
    item_to_sid = json.load(file)

expected_item_ids = {str(i) for i in range(len(items))}
sid_item_ids = set(item_to_sid)
sid_texts = [''.join(tokens) for tokens in item_to_sid.values()]
duplicate_sid_count = len(sid_texts) - len(set(sid_texts))

if expected_item_ids != sid_item_ids:
    raise ValueError(
        f'item.json 与 index.json 的 item_id 覆盖不一致：'
        f'缺失 {len(expected_item_ids - sid_item_ids)} 个，'
        f'多余 {len(sid_item_ids - expected_item_ids)} 个。'
    )

if duplicate_sid_count != 0:
    raise ValueError(f'index.json 中仍有 {duplicate_sid_count} 个重复完整 SID，暂不应转换。')

print('输入检查通过。')
print(f'商品数 / SID 映射数: {len(items):,} / {len(item_to_sid):,}')
print(f'完整 SID 冲突数: {duplicate_sid_count:,}')

for split_name, split_file in SPLIT_FILES.items():
    with split_file.open('r', encoding='utf-8') as file:
        sample_count = sum(1 for _ in file) - 1
    print(f'{split_name}.inter 样本数: {sample_count:,}')

print(f'输出目录: {OUTPUT_DIR}')

输入检查通过。
商品数 / SID 映射数: 3,106 / 3,106
完整 SID 冲突数: 0
train.inter 样本数: 29,458
valid.inter 样本数: 3,682
test.inter 样本数: 3,683
输出目录: D:\转码ing\MiniOneRec-main\data\Amazon18


In [3]:
# 确认上一个单元通过后，执行 item ID 到 SID 的格式转换。
command = [
    sys.executable, str(CONVERT_SCRIPT),
    '--dataset_name', DATASET,
    '--data_dir', str(DATA_DIR),
    '--output_dir', str(OUTPUT_DIR),
    '--category', CATEGORY,
    '--seed', str(SEED),
]

if MAX_VALID_SAMPLES is not None:
    command.extend(['--max_valid_samples', str(MAX_VALID_SAMPLES)])

if MAX_TEST_SAMPLES is not None:
    command.extend(['--max_test_samples', str(MAX_TEST_SAMPLES)])

print('将执行命令：')
print(' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

将执行命令：
c:\Users\k\.conda\envs\tf_env\python.exe D:\转码ing\MiniOneRec-main\convert_dataset.py --dataset_name Industrial_and_Scientific --data_dir D:\转码ing\MiniOneRec-main\data\Amazon18_2016_10_2018_11\Industrial_and_Scientific --output_dir D:\转码ing\MiniOneRec-main\data\Amazon18 --category Industrial_and_Scientific --seed 42


CompletedProcess(args=['c:\\Users\\k\\.conda\\envs\\tf_env\\python.exe', 'D:\\转码ing\\MiniOneRec-main\\convert_dataset.py', '--dataset_name', 'Industrial_and_Scientific', '--data_dir', 'D:\\转码ing\\MiniOneRec-main\\data\\Amazon18_2016_10_2018_11\\Industrial_and_Scientific', '--output_dir', 'D:\\转码ing\\MiniOneRec-main\\data\\Amazon18', '--category', 'Industrial_and_Scientific', '--seed', '42'], returncode=0)

In [4]:
# 检查 CSV 和 SID—标题—item_id 信息表，并仅展示少量样本。
import ast
import csv

OUTPUT_FILE_NAME = f'{CATEGORY}_5_2016-10-2018-11.csv'
CSV_FILES = {
    split_name: OUTPUT_DIR / split_name / OUTPUT_FILE_NAME
    for split_name in ['train', 'valid', 'test']
}
INFO_FILE = OUTPUT_DIR / 'info' / f'{CATEGORY}_5_2016-10-2018-11.txt'

expected_outputs = [*CSV_FILES.values(), INFO_FILE]
missing_outputs = [path for path in expected_outputs if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError('缺少以下转换输出：\n' + '\n'.join(map(str, missing_outputs)))

print('=== 输出文件 ===')
for path in expected_outputs:
    print(f'{path.relative_to(OUTPUT_DIR)}: {path.stat().st_size / 1024 / 1024:.2f} MiB')

for split_name, csv_file in CSV_FILES.items():
    with csv_file.open('r', encoding='utf-8', newline='') as file:
        reader = csv.DictReader(file)
        sample_rows = []
        total_count = 0

        for row in reader:
            total_count += 1
            if len(sample_rows) < 2:
                sample_rows.append(row)

    print(f'\n=== {split_name} ===')
    print(f'样本数: {total_count:,}')

    for row in sample_rows:
        print({
            'user_id': row['user_id'],
            'history_item_id': ast.literal_eval(row['history_item_id']),
            'item_id': row['item_id'],
            'history_item_sid': ast.literal_eval(row['history_item_sid']),
            'item_sid': row['item_sid'],
            'item_title': row['item_title'][:100],
        })

with INFO_FILE.open('r', encoding='utf-8') as file:
    info_line_count = sum(1 for _ in file)

print('\n=== info 文件 ===')
print(f'SID—标题—item_id 行数: {info_line_count:,}')
with INFO_FILE.open('r', encoding='utf-8') as file:
    for _ in range(3):
        print(file.readline().rstrip())

=== 输出文件 ===
train\Industrial_and_Scientific_5_2016-10-2018-11.csv: 15.17 MiB
valid\Industrial_and_Scientific_5_2016-10-2018-11.csv: 2.30 MiB
test\Industrial_and_Scientific_5_2016-10-2018-11.csv: 2.43 MiB
info\Industrial_and_Scientific_5_2016-10-2018-11.txt: 0.31 MiB

=== train ===
样本数: 29,458
{'user_id': 'A3545', 'history_item_id': [245], 'item_id': '245', 'history_item_sid': ['<a_228><b_204><c_218>'], 'item_sid': '<a_228><b_204><c_218>', 'item_title': 'Rubbermaid Commercial BRUTE Heavy-Duty Round Waste/Utility Container with Venting Channels, 20-gallo'}
{'user_id': 'A4150', 'history_item_id': [939], 'item_id': '939', 'history_item_sid': ['<a_17><b_225><c_37>'], 'item_sid': '<a_17><b_225><c_37>', 'item_title': '360-piece Solderless Electrical Terminal Assortment'}

=== valid ===
样本数: 3,682
{'user_id': 'A4771', 'history_item_id': [1644, 1688, 1710, 1289], 'item_id': '2796', 'history_item_sid': ['<a_125><b_53><c_18>', '<a_220><b_36><c_55>', '<a_53><b_247><c_173>', '<a_79><b_173><c_176>'